## Phase 1 : Document Loader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

C:\Users\gagan\AppData\Local\Temp\ipykernel_12576\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader("data/doc1.pdf")
documents = loader.load()

In [3]:
print("Pages : ", len(documents))

Pages :  2


In [4]:
for doc in documents[:2]:
    print(doc.page_content[:500])

General
 
Disease–Medicine
 
Dataset
 
1.
 
Common
 
Cold
 
○
 
Symptoms:
 
Runny
 
nose,
 
sneezing,
 
sore
 
throat,
 
mild
 
fever
 
○
 
Common
 
medicines:
 
Paracetamol,
 
saline
 
nasal
 
spray
 
○
 
Note:
 
Usually
 
resolves
 
with
 
rest
 
and
 
fluids.
 
2.
 
Fever
 
○
 
Symptoms:
 
Increased
 
body
 
temperature,
 
chills,
 
weakness
 
○
 
Common
 
medicine:
 
Paracetamol
 
○
 
Note:
 
Dosage
 
depends
 
on
 
age,
 
weight,
 
and
 
medical
 
condition.
 
3.
 
Headache
 
○
 
Symptoms:

11.
 
Migraine
 
○
 
Symptoms:
 
Severe
 
headache,
 
nausea,
 
sensitivity
 
to
 
light/sound
 
○
 
Common
 
medicines:
 
Paracetamol,
 
Ibuprofen
 
○
 
Note:
 
Recurrent
 
migraines
 
may
 
require
 
prescription
 
treatment.
 
12.
 
Motion
 
Sickness
 
○
 
Symptoms:
 
Nausea,
 
dizziness,
 
vomiting
 
during
 
travel
 
○
 
Common
 
medicine:
 
Dimenhydrinate
 
○
 
Note:
 
Some
 
medicines
 
can
 
cause
 
drowsiness.
 
13.
 
Athlete's
 
Foot
 
○
 
Symptoms:
 
Itching,
 
scaling,
 
cracking
 


In [5]:
print(doc.metadata)

{'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'doc1', 'source': 'data/doc1.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}


## Phase 2 : Chunk the document

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

chunks = splitter.split_documents(documents)

print("Original Pages : ", len(documents))
print("Chunks : ", len(chunks))

Original Pages :  2
Chunks :  19


## Phase 3 : Embeddings

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\gagan\Desktop\GenAI\Module 9\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4754.98it/s]


In [10]:
vector = embeddings.embed_query("Which medicine i can take in Acidity?")
print(vector, len(vector))

[0.054265592247247696, -0.06661278009414673, -0.045572489500045776, 0.023062242195010185, 0.0016891960985958576, 0.028440287336707115, 0.016607291996479034, 0.11546022444963455, -0.0029458284843713045, -0.09081505984067917, -0.0013142168754711747, 0.00898254755884409, 0.012023944407701492, 0.009925393387675285, -0.0017142584547400475, 0.05332154035568237, 0.02856582961976528, 0.03522801399230957, -0.03653639927506447, -0.0630565732717514, -0.06846828013658524, 0.01056759525090456, 0.012519536539912224, 0.0036376917269080877, -0.03964126110076904, 0.06860560923814774, 0.04362974688410759, -0.026511218398809433, -0.004376389551907778, -0.11233854293823242, 0.11417580395936966, -0.002000807784497738, 0.021817361935973167, -0.026693550869822502, -0.06152435764670372, 0.012798015028238297, -0.1133938580751419, -0.057950347661972046, 0.08347533643245697, 0.05038154870271683, 0.04242831468582153, -0.015629081055521965, -0.00019377487478777766, -0.06682536751031876, 0.027823947370052338, -0.07

## Phase 4 : Chroma Database (storing vector)

In [11]:
from langchain_chroma import Chroma

In [12]:
vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

## Phase 5 : Retriever

In [13]:
retriever = vectorStore.as_retriever(
    search_kwargs={"k":4}
)

In [ ]:
query = "Medicine for headache?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print("Results : ", i+1)
    print(doc.page_content)
    print(doc.metadata)
    print("--------")

Results :  1
the
 
head
 
○
 
Common
 
medicines:
 
Paracetamol,
 
Ibuprofen
 
○
 
Note:
 
Frequent
 
or
 
severe
 
headaches
 
require
 
medical
 
evaluation.
 
4.
 
Allergic
 
Rhinitis
 
○
 
Symptoms:
 
Sneezing,
 
itchy
 
nose,
 
runny
 
nose
 
○
 
Common
 
medicines:
 
Cetirizine,
 
Loratadine
 
○
 
Note:
{'title': 'doc1', 'page': 0, 'creator': 'PyPDF', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'creationdate': '', 'page_label': '1', 'source': 'data/doc1.pdf', 'total_pages': 2}
--------
Results :  2
the
 
head
 
○
 
Common
 
medicines:
 
Paracetamol,
 
Ibuprofen
 
○
 
Note:
 
Frequent
 
or
 
severe
 
headaches
 
require
 
medical
 
evaluation.
 
4.
 
Allergic
 
Rhinitis
 
○
 
Symptoms:
 
Sneezing,
 
itchy
 
nose,
 
runny
 
nose
 
○
 
Common
 
medicines:
 
Cetirizine,
 
Loratadine
 
○
 
Note:
{'total_pages': 2, 'page_label': '1', 'source': 'data/doc1.pdf', 'title': 'doc1', 'page': 0, 'creator': 'PyPDF', 'producer': 'Skia/PDF m156 Google Docs Renderer', 'creationdate': ''}
---

## Phase 6 : Connect LLM

In [15]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="mistral",
    temperature=0.5
)

## Phase 7 : Create Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(""" 
You are a document question-answering assistant.

Answer the question using Only the provided context.

If the answer is not present in the context,
say : "I Could not find this info in documents.

Context : {context}
question : {question}

answer : 
""")

## Phase 8 : Generate Answer

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in results
)

messages = prompt.invoke({
    "context" : context,
    "question": query
})

answer = llm.invoke(messages)

print(answer)

 Paracetamol, Ibuprofen


In [28]:
def ask_query(question : str):
    """ Retrives context and generates answer for a single query."""

    results = retriever.invoke(question)
    context =  "\n\n".join(doc.page_content for doc in results)

    messages = prompt.invoke({
        "context" : context,
        "question": question
    })
    
    response = llm.invoke(messages)

    print(f"Question : {question}\n")
    print(f"Answer:\n{response}\n")

    print("Source : ")
    sources = set(
        f"- {doc.metadata.get('source')} (page {doc.metadata.get('page')+1})"
        for doc in results
    )
    for src in sources:
        print(sources)


In [29]:
ask_query("which medicine i can take in acidity?")

Question : which medicine i can take in acidity?

Answer:
 Antacids and Famotidine are common medicines for treating acidity/heartburn, as mentioned in the provided context.

Source : 
{'- data/doc1.pdf (page 1)'}


In [31]:
ask_query("which medicine i can take in Hypertension and what are symtoms?")

Question : which medicine i can take in Hypertension and what are symtoms?

Answer:
 In Hypertension, common medicines include Amlodipine, Losartan, and other antihypertensives. The symptoms of hypertension often have no symptoms. However, it's important to note that the medication must be selected and adjusted by a healthcare professional due to the requirement for diagnosis and ongoing medical supervision.

Source : 
{'- data/doc1.pdf (page 1)', '- data/doc1.pdf (page 2)'}
{'- data/doc1.pdf (page 1)', '- data/doc1.pdf (page 2)'}
